# Part 5. 평가와 모니터링의 결합으로 지속 운영하기

이 노트북은 Day05_new의 여섯 번째 노트북입니다. 앞에서는 평가(01~03b)와 모니터링(04)을 따로 배웠습니다. 여기서는 그 둘을 **한 바퀴 도는 운영 루프**로 묶습니다. 운영 trace에서 사람이 볼 표본을 고르고, 사람이 본 실패를 회귀 Golden으로 승격하고, 평가기를 바꿨을 때 두 Agent 버전을 같은 기준으로 다시 채점하고, 마지막에 배포해도 되는지 승인 게이트로 판단하는 흐름입니다.

### 수업 시간표

| 구간 | 내용 | 분 |
|---|---|---|
| 0. 준비 | 가져오기, 작업 폴더, 키 상태, 산출물 폴더 | 3 |
| 1. 운영 루프 한 장 | 루프의 다섯 단계와 `eval_lab` 함수의 대응 | 4 |
| 2. 두 버전의 원자료와 채점 | 동결본 한 벌, baseline·candidate 원자료, 현재 평가기로 채점 | 6 |
| 3. 표본 선정과 annotation queue | 운영 트래픽, 선정 전략, 사람 검수 대기열과 CSV 대체 경로 | 7 |
| 4. 회귀 Golden 승격 | 실패 기록을 미검수 초안으로, 기대 상태는 사람이 채운다 | 6 |
| 5. 평가기 변경 관리 | 두 버전을 같은 새 평가기로 재채점, 과거 trace에 backtest | 10 |
| 6. 승인 게이트 | 동일 조건 비교와 PREREG P5의 모양 | 6 |
| 7. 직접 해보기 | 검수 칸 채우기, 전략 비교, 의미 판정 연쇄, P5 행 고쳐 보기 | 3 |

### 이 노트북이 하는 일 (그리고 하지 않는 일)

- 하는 일: 이미 만들어 둔 부품(`eval_lab`의 함수들)을 **루프의 순서대로 연결**합니다. 새 평가 원리를 배우지 않습니다. 선정, 대기열, 승격, 재채점, 게이트가 각각 어느 함수이고 어디서 멈추는지 직접 실행해서 확인합니다.
- 하지 않는 일: 에이전트를 실제 모델로 돌리지 않습니다. 이 노트북의 Episode는 모두 모델 없이 도구만 부르는 scripted 기록이라 **모의 운영 트래픽이며 모델 성능이 아닙니다.** 그래서 어떤 표의 PASS 수나 HOLD 판정도 "Agent가 좋다/나쁘다"의 증거가 아닙니다. 연결이 맞게 되는지, 규칙이 어디서 멈추는지를 보는 용도입니다. 자동 배포도 하지 않습니다. 게이트는 사람이 판단할 근거를 모으는 장치입니다.

### 호출 규모

- 0~4절, 5절 앞부분, 6절과 직접 해보기의 기본 셀: 유료 호출이 없습니다. Claude도 Jev도 Judge도 부르지 않습니다.
- 5절의 backtest 셀 하나: Jev 최대 20회(과거 trace 20건에 한 건당 한 번), Judge 0회, Claude 0회입니다. 새 평가기를 `jev_only` 모드로 돌리므로 Judge는 구조상 부를 수 없습니다. Jev가 확신하지 못한 질문은 판정을 보류로 남깁니다. 이 비용은 본인의 Decision API 크레딧에서 나가고, 실행 직전에 셀이 같은 내용을 한 줄로 다시 알립니다.
- 직접 해보기 3의 `RUN_OPTIONAL` 셀 하나: Jev 최대 5회, Judge(xhigh) 최대 15회입니다. 기본값이 꺼져 있어서 켜지 않으면 아무것도 부르지 않습니다.
- Langfuse 키가 없으면 로컬 경로로 진행합니다. 같은 평가기를 로컬 기록에 돌리므로 호출 규모는 같습니다.

### 표지 규칙

코드 셀 첫 줄의 표지는 하는 일을 알려 줍니다. `# [따라하기]`는 그대로 실행, `# [바꿔보기]`는 값을 바꿔 다시 실행, `# [깨뜨리기]`는 일부러 규칙을 어겨 어디서 멈추는지 보는 셀, `# [내 것 만들기]`는 내가 직접 채우는 셀입니다. `**먼저 예측하세요.**`가 나오면 실행하기 전에 결과를 한 줄이라도 적어 보세요. 맞히는 것이 목적이 아니라, 틀렸을 때 어느 가정이 틀렸는지 알아내는 것이 목적입니다. `**관찰할 것.**`은 출력에서 무엇을 읽어야 하는지 알려 주는 문단입니다.

이 저장소의 용어를 그대로 씁니다. 근거(Evidence), 원문, 적재(ingest), 담당자(owner), 예산(budget), 인용, 기준일(as_of), 고객(entity)입니다. 문서와 코드가 다르면 코드가 정답입니다.

## 0. 준비

먼저 부품을 가져오고, 커널의 작업 폴더가 `Day05_new`인지 확인합니다. 상대 경로(`data/`, `eval_lab/`)와 `.env`가 이 폴더 기준이기 때문입니다. `show_keys`는 키가 **설정됨/미설정**인지만 보여 주고 값은 보여 주지 않습니다.

`langfuse_or_none`은 Langfuse 키가 있으면 클라이언트를 돌려주고, 없으면 안내 한 줄을 출력하고 `None`을 돌려줍니다. 이 노트북의 모든 셀은 `client`가 있는 경로와 없는 경로가 **둘 다 끝까지 돌아가도록** 쓰여 있습니다. 모델과 Jev는 `eval_lab.settings`의 `make_*` 함수와 `eval_lab.decisions.make_decider`를 통해서만 만듭니다. 노트북 안에서 `ChatAnthropic`을 직접 만들지 않습니다. 키와 모델 이름을 한 곳에서 관리해야 비용 고지와 기록이 어긋나지 않기 때문입니다.

`new_run_dir("05")`는 이 실행만의 새 폴더를 `outputs/notebook/05/` 아래에 만듭니다. 이 노트북이 쓰는 파일은 모두 그 폴더 안에만 생깁니다.

In [ ]:
# [따라하기] 가져오기, 작업 폴더 확인, 키 상태, 이 실행의 산출물 폴더
import importlib.util
import inspect
import json
from collections import Counter
from pathlib import Path

from eval_lab import decisions
from eval_lab import settings as lab_settings
from eval_lab.agent import PROMPTS
from eval_lab.authoring import load_frozen, read_workbook
from eval_lab.contracts import Scenario
from eval_lab.dataset import DATA, fingerprint, load_dataset, load_fixtures
from eval_lab.evidence import export_human_review, promote_failure
from eval_lab.notebook import announce, langfuse_or_none, new_run_dir, show, show_keys
from eval_lab.online import (
    batch_evaluate,
    enqueue_for_review,
    evaluate_records_locally,
    online_evaluators,
    request_shape,
    select_traces,
)
from eval_lab.reporting import status_of
from eval_lab.rsi import ADOPT, HOLD, NOT_DETERMINABLE, holdout_decision, holdout_row
from eval_lab.runs import compare_scored, grade_recording, record_release
from eval_lab.storage import make_run_id, save_run_report
from eval_lab.traffic import make_mistake_driver, make_traffic, run_traffic
from langchain_core.language_models.chat_models import BaseChatModel

print("커널 작업 폴더:", Path.cwd().name)
assert Path("eval_lab").is_dir(), "커널의 작업 폴더가 Day05_new여야 합니다."

# 1단계: 키가 설정됐는지만 본다.
settings = lab_settings.load_settings()
show_keys(settings)

# 2단계: Langfuse 클라이언트. 없으면 None이고, 이후 셀은 로컬 경로로 간다.
client = langfuse_or_none(settings)
print("Langfuse 연결:", "있음" if client is not None else "없음(로컬 경로)")

# 3단계: 이 실행의 산출물 폴더. 이 노트북은 이 폴더 안에만 파일을 쓴다.
run_dir = new_run_dir("05")
print("산출물 폴더:", run_dir)

## 1. 운영 루프 한 장

평가와 모니터링을 따로 두면 두 가지가 비어 버립니다. 모니터링은 "무언가 달라졌다"는 신호는 주지만 **무엇을 성공으로 볼지**는 모릅니다. 오프라인 평가는 성공 기준은 갖고 있지만 **운영에서 새로 생긴 실패**는 모릅니다. 운영 루프는 이 둘을 서로의 입력으로 연결합니다.

```
운영 trace ─> 온라인 평가(Code→Jev→Judge) ─┐
     │                                   ├─> 표본 선정 ─> annotation queue(사람)
     └─> 모니터링 지표·경보 ───────────────┘                    │
                                                               v
   배포 <─ 승인 게이트 <─ 오프라인 실험(dev) <─ 회귀 Golden 승격(새 데이터 버전)
   (사전 등록·holdout·가드)
```

화살표를 순서대로 읽으면 이렇습니다.

1. **운영 trace가 쌓입니다.** 정답이 없는 기록입니다. 그래서 온라인 평가는 정답 없이 되는 것(궤적 규칙, 의미 판정)만 씁니다(03a, 03b).
2. **온라인 평가와 모니터링이 각자 신호를 냅니다.** 평가는 "이 trace는 규칙을 어겼다"를, 모니터링은 "어제보다 호출 수가 늘었다"를 말합니다(04).
3. **표본을 골라 사람에게 보냅니다.** 전부 볼 수는 없으므로 전략을 정해 고릅니다. 사람이 본 결과가 평가기를 믿을 수 있는지의 기준이 됩니다.
4. **사람이 확인한 실패는 회귀 Golden으로 승격합니다.** 단, 기대 상태는 비워 두고 사람이 채웁니다. 승격한 카드는 **새 데이터 버전**에서만 씁니다.
5. **오프라인 실험으로 고친 Agent를 확인하고, 승인 게이트를 통과해야 배포합니다.** 게이트는 미리 등록한 규칙(PREREG)대로 판정합니다.

이 노트북은 이 다섯 칸을 한 번씩 직접 건드립니다. 먼저 각 칸을 맡은 함수가 `eval_lab`의 어디에 있는지 코드가 직접 찾아서 보여 줍니다. 표의 위치는 제가 적은 것이 아니라 `inspect`가 읽은 값입니다.

In [ ]:
# [따라하기] 루프의 단계와 그것을 맡은 함수의 위치를 코드가 직접 찾는다
LOOP = [
    ("3. 표본 선정", select_traces),
    ("3. 사람 검수 대기열", enqueue_for_review),
    ("3. 대기열 대체 경로(CSV)", export_human_review),
    ("4. 회귀 Golden 승격", promote_failure),
    ("5. 같은 기준으로 재채점", grade_recording),
    ("5. 과거 trace에 새 평가기 backtest", batch_evaluate),
    ("6. 동일 조건 비교", compare_scored),
    ("6. holdout 판정(PREREG P5)", holdout_decision),
]
rows = []
for stage, function in LOOP:
    source = Path(inspect.getsourcefile(function))
    line = inspect.getsourcelines(function)[1]  # 함수가 시작하는 줄 번호
    rows.append({"루프 단계": stage, "함수": function.__name__, "위치": f"{source.parent.name}/{source.name}:{line}"})
show(rows)

**관찰할 것.** 위치 칸을 보면 다섯 단계가 `online.py`, `evidence.py`, `runs.py`, `rsi.py` 네 파일에 나뉘어 있습니다. 평가기가 하는 일(`online.py`)과 사람이 개입하는 일(`evidence.py`), 두 버전을 비교하는 일(`runs.py`), 배포를 판단하는 일(`rsi.py`)이 파일 단위로도 분리되어 있다는 뜻입니다. 분리되어 있어야 한 칸을 바꿨을 때 나머지 칸의 결과가 왜 달라졌는지 설명할 수 있습니다.

`inspect`가 읽은 줄 번호는 파일이 바뀌면 같이 바뀝니다. 문서에 적힌 줄 번호와 이 표가 다르면 **이 표가 정답입니다.**

## 2. 두 버전의 원자료와 채점

루프 4~6칸은 모두 "두 Agent 버전을 같은 조건에서 비교한다"는 같은 재료를 씁니다. 그 재료를 먼저 만들어 둡니다.

- **동결본.** dev에서 검수·승인된 카드를 한 묶음으로 고정하고 해시(`snapshot_hash`)를 찍은 파일입니다. 동결 뒤에 내용이 바뀌면 `load_frozen`(`eval_lab/authoring.py:168`)이 읽기를 거부합니다. 같은 데이터로 실행했다는 증거가 이 해시입니다.
- **원자료(recording).** 한 버전을 실행한 기록입니다. 실행과 채점을 **분리**하는 것이 핵심입니다. 채점기를 바꿨을 때 Agent를 다시 돌리지 않고 같은 기록을 다시 채점할 수 있어야 하기 때문입니다(`record_release`, `eval_lab/runs.py:110`).
- **채점본(scored recording).** 원자료에 판정을 붙인 복사본입니다(`grade_recording`, `eval_lab/runs.py:199`). 원자료는 건드리지 않습니다.

두 버전은 `eval_lab.agent.PROMPTS`의 `baseline`과 `candidate`입니다. candidate는 baseline 프롬프트에 "환불이 TIMEOUT이면 먼저 결제 상태를 조회하라"는 복구 지침을 더한 것입니다. 이 노트북은 scripted 드라이버로 실행하므로 프롬프트 문장이 아니라 **드라이버가 따르는 분기**(`eval_lab/agent.py`의 `run_scripted`)가 두 버전의 차이를 만듭니다. 모델이 실제로 그렇게 행동한다는 보장은 없습니다.

**먼저 예측하세요.** 같은 동결본 9개 사례를 두 버전이 각각 한 번씩 실행하면, 업무 결과가 갈리는 사례가 몇 개쯤 나올까요? 어느 장애 유형(timeout, payment-down, shipping-race 등)에서 갈릴 것 같은지도 한 줄 적어 보세요.

In [ ]:
# [따라하기] dev의 승인된 카드를 동결본으로 만든다 (02에서 한 동결과 같은 문서 모양)
def make_frozen(cards: list[dict], fixtures: dict, out: Path) -> dict:
    """카드 목록을 스냅샷으로 묶고 해시를 찍어 파일로 저장한다. 같은 내용이면 같은 해시가 나온다."""
    snapshot = {
        "cards": cards,
        "fixtures": {card["fixture_id"]: fixtures[card["fixture_id"]] for card in cards},
        "policy": (DATA / "order_policy.md").read_text(),
        "schema_version": "course-v2",
        "split": "dev",
    }
    document = {
        "run_id": make_run_id(),
        "kind": "approved_dataset",
        "snapshot_hash": fingerprint(snapshot),
        "snapshot": snapshot,
        "coverage": dict(Counter(card["metadata"]["slice"] for card in cards)),
        "reviewed_ids": [card["metadata"]["scenario_id"] for card in cards],
        "not_selected_ids": [],
    }
    save_run_report(document, out)
    return document


dev_cards = [card.model_dump() for card in load_dataset(split="dev")]  # 검수가 끝난 dev 카드만 읽힌다
fixtures = load_fixtures()
make_frozen(dev_cards, fixtures, run_dir / "dev-v1.frozen.json")
frozen = load_frozen(run_dir / "dev-v1.frozen.json")  # 해시가 맞아야 읽힌다

print("동결본 해시(앞 12자):", frozen["snapshot_hash"][:12])
print("카드 수:", len(frozen["snapshot"]["cards"]), "/ 분할:", frozen["snapshot"]["split"])
show([{"slice": name, "카드 수": count} for name, count in frozen["coverage"].items()])

In [ ]:
# [따라하기] baseline과 candidate를 같은 동결본으로 한 번씩 실행해 원자료로 저장한다 (모델 호출 0회)
raws = {}
for release in ("baseline", "candidate"):
    raws[release] = record_release(
        frozen, release=release, prompt=PROMPTS[release], repeats=1, mode="scripted",
        out=run_dir / f"{release}.recording.json", langfuse=client,
    )

show([{"버전": release, "상태": raw["status"], "시도 수": len(raw["rows"]), "원자료 해시(앞 12자)": raw["recording_hash"][:12]}
      for release, raw in raws.items()])

In [ ]:
# [따라하기] 현재 평가기(v1)로 두 원자료를 채점하고 판정 개수를 센다
def verdict_table(scored_by_release: dict) -> list[dict]:
    """버전마다 판정 항목별 PASS/FAIL/불충분 개수와, 시도 전체의 종합 상태 개수를 센다."""
    table = []
    for release, scored in scored_by_release.items():
        counts = Counter((v["name"], v["status"]) for row in scored["rows"] for v in row["verdicts"])
        for name in sorted({name for name, _ in counts}):
            table.append({"버전": release, "판정 항목": name, "PASS": counts[name, "PASS"],
                          "FAIL": counts[name, "FAIL"], "불충분·오류": counts[name, "INSUFFICIENT_EVIDENCE"]
                          + counts[name, "EVALUATOR_ERROR"]})
        overall = Counter(status_of(row) for row in scored["rows"])
        table.append({"버전": release, "판정 항목": "(종합 상태)", "PASS": overall["PASS"], "FAIL": overall["FAIL"],
                      "불충분·오류": sum(overall.values()) - overall["PASS"] - overall["FAIL"]})
    return table


scored_v1 = {release: grade_recording(raw) for release, raw in raws.items()}
show(verdict_table(scored_v1))

**관찰할 것.** 판정 항목이 `outcome`(최종 업무 상태), `side_effects`, `policy`, `recovery`, `response_contract`, `milestones`, `episode`로 나뉘어 있고, 종합 상태는 이 항목들을 모아서 정합니다(`status_of`, `eval_lab/reporting.py:12`). 두 버전의 표를 위아래로 비교해서 어느 항목에서 개수가 다른지 찾아 보세요. 예측과 비교해서 어느 가정이 틀렸는지도 적어 보세요. 개수가 같다고 같은 사례에서 같은 결과가 난 것은 아닙니다. 사례 단위 비교는 6절에서 합니다.

위 판정에는 **Judge가 없습니다.** scripted 실행은 응답 문장이 정해진 문구라서 의미 판정을 하지 않고, `grade_recording`이 `judge=None`으로 돌았습니다. 지금 보는 것은 Code 판정(상태, 정책, 순서)뿐입니다. 이 채점기를 "평가기 v1"이라고 부르겠습니다. 5절에서 이 평가기를 바꿉니다.

## 3. 표본 선정과 annotation queue

운영에서는 trace가 하루에 수천 건 쌓일 수 있고 사람이 볼 수 있는 것은 수십 건입니다. **무엇을 사람에게 보낼지**가 곧 평가기를 믿는 방식을 정합니다. `select_traces`(`eval_lab/online.py:147`)는 네 가지 전략을 지원하고, 각 전략은 놓치는 것이 다릅니다.

- `random`: 공평하지만 드문 문제를 놓칩니다. 전체의 2%만 문제라면 k건 중 평균 0.02k건만 잡힙니다.
- `stratified`: (가족, 날짜) 층마다 돌아가며 한 건씩 뽑습니다. 층 안의 발생률 차이와, 분류에 없는 새 유형을 놓칩니다.
- `signal`: 도구 오류, 모델 오류, escalated, 지연·비용 상위, 규칙 위반 순서로 뽑습니다. 신호 없이 조용히 틀린 trace를 놓칩니다. **편향된 표본이라 전체 비율을 말하는 근거로 쓰면 안 됩니다.**
- `novelty`: 요청 모양이 Golden에 없는 것만 뽑습니다. Golden에 있는 모양에서 생긴 회귀를 놓칩니다. 비교할 Golden이 없으면 에러입니다.

여기서는 운영 트래픽 24건을 만들어 전략을 적용합니다. `make_traffic`은 날짜별 분포로 요청을 만들고, `make_mistake_driver`는 scripted 드라이버가 일부 Episode에서 **실제 도구 호출로** 실수를 섞게 합니다(승인 전 쓰기, 기록 없는 완료 주장). 말로만 적은 실수가 아니라 Event와 업무 상태에 남는 실수라서 궤적 규칙이 잡을 수 있습니다. 이것도 **모의 트래픽이며 LLM 성능이 아닙니다.**

Episode마다 Langfuse의 root observation이 하나 생기고, 환경(environment) 이름과 버전 라벨(`version="policy-v1"`)이 붙습니다. 환경 이름은 나중에 "어느 trace를 읽을지" 고르는 필터가 됩니다. 5절의 backtest가 바로 그 필터로 root observation을 읽습니다(5절은 별도의 환경 이름을 씁니다).

In [ ]:
# [따라하기] 모의 운영 트래픽 24건을 실행한다 (모델 호출 0회, 실수를 일부러 섞은 scripted 기록)
OPERATION_ENV = "day05-op-live"

items = make_traffic(1, 24, seed=5)
driver = make_mistake_driver({"write_before_approval": 0.2, "claim_without_record": 0.2}, seed=11)
records = run_traffic(items, mode="scripted", langfuse=client, driver=driver,
                      environment_tag=OPERATION_ENV, release="baseline", version="policy-v1")
if client is not None:
    client.flush()  # 아직 보내지 않은 기록을 내보낸다

print("기록 수:", len(records))
print("실행 상태:", dict(Counter(record["execution_status"] for record in records)))
print("trace ID가 붙은 기록:", sum(record["trace_id"] is not None for record in records))
print("섞인 실수가 있는 기록:", sum(bool(record["injected_mistakes"]) for record in records))

In [ ]:
# [바꿔보기] 전략과 k를 바꿔 어떤 trace가 뽑히는지 본다
STRATEGY = "signal"  # "random", "stratified", "signal" 중 하나로 바꿔 보세요
K = 6

selections = select_traces(records, STRATEGY, K, seed=3)
show([{"기록 위치": s.record_index, "항목": records[s.record_index]["item_id"], "뽑힌 이유": s.reason}
      for s in selections], title=f"전략 {STRATEGY}, k={K}: {len(selections)}건 선정")

**관찰할 것.** `signal` 전략은 신호가 있는 기록만 뽑으므로 k보다 적게 돌려줄 수 있습니다. 뽑힌 이유 칸을 읽으면 어떤 신호(규칙 위반, 도구 오류, 지연 상위 등)가 우선순위를 정했는지 알 수 있습니다. `STRATEGY`를 `random`과 `stratified`로 바꿔 다시 실행해서, 같은 기록에서 뽑히는 항목이 어떻게 달라지는지 비교해 보세요. 같은 seed와 같은 입력이면 **항상 같은 결과**가 나오는 것도 확인할 수 있습니다. 사람이 본 표본을 나중에 다시 재현해야 하기 때문입니다.

다음은 뽑은 trace를 사람에게 보내는 칸입니다.

### annotation queue, 그리고 클라이언트가 없을 때

Langfuse가 연결되어 있으면 `enqueue_for_review`(`eval_lab/online.py:469`)가 대기열을 찾거나 만들고 선정한 trace를 넣습니다. `trace_id`가 없는 선정은 넣을 수 없으므로 **건너뛰되 개수를 세서** 돌려줍니다. 조용히 빼면 "다 보냈다"는 착각이 생깁니다.

Langfuse가 없으면 대기열이 없습니다. 이때의 대체 경로가 `export_human_review`(`eval_lab/evidence.py:114`)입니다. 채점본에서 **Judge 점수를 가린 빈 검수 CSV**와, 사람이 읽을 원자료 묶음(`.context.json`)을 만듭니다. 사람이 CSV의 `human_verdict`, `reviewer`, `reason` 칸을 채우면 나중에 평가기와 대조합니다(`eval_lab/review.py`). 대체 경로는 **선정한 표본이 아니라 채점본의 모든 시도**를 내보냅니다. 선정 결과를 CSV에 반영하는 일은 이 노트북이 하지 않는 일입니다.

In [ ]:
# [따라하기] 사람 검수 대기열에 넣는다. 클라이언트가 없으면 빈 검수 CSV를 만든다
if client is not None:
    queue_result = enqueue_for_review(client, "day05-new-review", selections)
    show([queue_result], title="annotation queue 결과")
else:
    csv_path = export_human_review(scored_v1["baseline"], run_dir / "human_review.csv")
    print("검수 원자료 묶음:", csv_path.name)
    review_csv = run_dir / "human_review.csv"
    print(review_csv.read_text(encoding="utf-8-sig").splitlines()[0])  # 사람이 채울 열 이름
    print("행 수(머리글 제외):", len(review_csv.read_text(encoding="utf-8-sig").splitlines()) - 1)

In [ ]:
# [깨뜨리기] 두 가지 실수: Golden 없이 "새로움"을 묻기, 사람이 채운 검수 파일을 덮어쓰기
try:
    select_traces(records, "novelty", 5, seed=1)
except ValueError as error:
    print("novelty, golden_shapes 없음 ->", error)

csv_path = run_dir / "break_review.csv"
export_human_review(scored_v1["candidate"], csv_path)  # 처음에는 만들어진다
csv_path.write_text("사람이 채운 내용이라고 가정", encoding="utf-8")
try:
    export_human_review(scored_v1["candidate"], csv_path)  # 같은 경로로 다시 내보내기
except FileExistsError as error:
    print("두 번째 내보내기 ->", type(error).__name__)
print("사람이 쓴 내용이 남아 있는가:", csv_path.read_text(encoding="utf-8") == "사람이 채운 내용이라고 가정")

**관찰할 것.** 첫 번째 실수는 에러 메시지가 "비교할 Golden이 없으면 새로움을 말할 수 없다"고 이유를 알려 줍니다. 기준이 없는 질문에는 기본값을 지어내지 않고 멈추는 것이 올바른 동작입니다. 두 번째 실수는 파일을 `x` 모드(없을 때만 생성)로 열기 때문에 **사람이 이미 써 둔 검수 결과를 덮어쓰지 못합니다.** 마지막 줄이 `True`인지 확인하세요. 사람의 판정은 재현할 수 없는 자료이고, 평가기를 믿는 기준이므로 코드가 지울 수 있게 두면 안 됩니다.

## 4. 회귀 Golden 승격

사람이 확인한 실패는 같은 일이 다시 생기지 않는지 지켜보는 **회귀 Golden**의 후보가 됩니다. `promote_failure`(`eval_lab/evidence.py:156`)가 하는 일은 세 가지입니다.

1. 실패한 Episode의 원래 Scenario 카드를 복사해 **새 `scenario_id`**(`regression-...`)를 붙이고, 출처 trace와 Artifact ID를 `metadata`에 남깁니다.
2. `review_status = "pending"`, 담당자(`reviewer`)와 근거(`review_reason`)는 비워 둡니다.
3. **기대 최종 상태(`expected_output.final_state`)를 비웁니다.** 실제 결과를 기대 상태에 복사하지 않습니다.

3번이 이 절의 핵심입니다. 실패한 Agent가 남긴 상태를 기대 상태로 적어 버리면 **틀린 행동이 정답으로 굳습니다.** 기대 상태는 정책과 초기 상태를 읽은 사람이 직접 적어야 합니다. `Expected.final_state`가 `min_length=1`이라서(`eval_lab/contracts.py:23`) 비어 있는 카드는 Scenario로 읽히지도 않습니다. 사람이 채우기 전에는 실행에 섞일 수 없다는 뜻입니다.

또 하나, `sanitization_record`(비식별화 확인 기록)가 비어 있으면 승격이 거부됩니다. 운영 trace에는 고객의 실제 정보가 들어 있을 수 있으므로 "보관해도 되는지 확인했다"는 사람의 기록을 요구합니다. 이 노트북의 데이터는 전부 교육용 가상 데이터라서 그렇게 적습니다.

승격한 카드는 기존 동결본(`dev-v1`)에 **끼워 넣지 않습니다.** 검수가 끝난 뒤 카드를 더해 만든 동결본은 해시가 다른 **새 데이터 버전**입니다. 이전 버전으로 잰 결과와 새 버전으로 잰 결과를 섞어 비교하지 않기 위해서입니다.

In [ ]:
# [따라하기] 실패한 시도 하나를 회귀 Golden 초안으로 승격한다
def first_failing_artifact_id(scored: dict) -> str:
    """종합 상태가 FAIL인 첫 시도의 artifact_id. FAIL이 하나도 없으면 첫 시도를 쓴다."""
    for row in scored["rows"]:
        if status_of(row) == "FAIL":
            return row["artifact"]["artifact_id"]
    return scored["rows"][0]["artifact"]["artifact_id"]


chosen_id = first_failing_artifact_id(scored_v1["baseline"])
chosen = next(row for row in raws["baseline"]["rows"] if row["artifact"]["artifact_id"] == chosen_id)
print("승격할 시도:", chosen["artifact"]["scenario_id"], "/ 종합 상태:",
      status_of(next(row for row in scored_v1["baseline"]["rows"] if row["artifact"]["artifact_id"] == chosen_id)))

draft = promote_failure(
    raws["baseline"], {"artifact_id": chosen_id}, run_dir / "regression-draft.xlsx",
    sanitization_record="교육용 가상 데이터. 고객 식별 정보와 실제 주문 번호가 없음을 직접 확인함.",
)
meta = draft["metadata"]
show([{"항목": "새 scenario_id", "값": meta["scenario_id"]},
      {"항목": "출처 Artifact", "값": meta["source_artifact_id"]},
      {"항목": "review_status", "값": meta["review_status"]},
      {"항목": "reviewer / review_reason", "값": f"{meta['reviewer']!r} / {meta['review_reason']!r}"},
      {"항목": "expected_output.final_state", "값": draft["expected_output"]["final_state"]}])
print("만들어진 파일:", sorted(path.name for path in run_dir.glob("regression-draft*")))

In [ ]:
# [따라하기] 사람이 열어 볼 검수 파일을 다시 읽어 상태를 확인한다
reloaded = read_workbook(run_dir / "regression-draft.xlsx")
print("검수 파일의 카드 수:", len(reloaded), "/ 상태:", reloaded[0]["metadata"]["review_status"])

fixture_file = json.loads((run_dir / "regression-draft.fixture.json").read_text(encoding="utf-8"))
print("초기 상태 근거 파일의 키:", sorted(fixture_file))
print("안내:", fixture_file["instruction"])

In [ ]:
# [깨뜨리기] 기대 상태가 빈 카드와 비식별화 기록이 없는 승격은 어디서 멈추는가
try:
    Scenario.model_validate(draft)
except ValueError as error:
    print("빈 기대 상태로 Scenario를 만들면 ->", str(error).splitlines()[0])
    print("  ", str(error).splitlines()[1].strip() if len(str(error).splitlines()) > 1 else "")

try:
    promote_failure(raws["baseline"], {"artifact_id": chosen_id}, run_dir / "no-record.xlsx", sanitization_record="   ")
except ValueError as error:
    print("비식별화 기록이 비어 있으면 ->", error)
print("거부된 승격이 파일을 남겼는가:", (run_dir / "no-record.xlsx").exists())

**관찰할 것.** 첫 번째는 pydantic이 `final_state`가 비어 있다는 이유로 카드를 거부합니다. 그래서 승격 초안은 **사람이 기대 상태를 채우기 전까지** `record_release`나 동결 단계로 흘러가지 못합니다. 두 번째는 비식별화 기록을 사람이 쓰지 않으면 승격 자체가 멈추고 **파일도 남기지 않습니다.** 마지막 줄이 `False`인지 확인하세요. 거부된 승격이 반쯤 쓴 파일을 남기면 나중에 누군가 그 파일을 그대로 쓸 수 있기 때문입니다.

방금 승격한 카드는 아직 어느 데이터 버전에도 들어 있지 않습니다. 직접 해보기 1에서 검수 칸을 직접 채워 새 데이터 버전을 만들어 봅니다.

## 5. 평가기 변경 관리

운영을 오래 하면 평가기도 바뀝니다. 새 규칙이 생기고, 놓치던 실패를 잡는 검사가 추가됩니다. 평가기를 바꿀 때 지켜야 할 규칙이 둘 있습니다.

**규칙 1: 두 Agent 버전의 원자료를 같은 새 평가기로 다시 채점한다.** 평가기가 바뀌었는데 baseline은 옛 채점 결과를, candidate는 새 채점 결과를 쓰면 점수 차이가 Agent의 차이인지 평가기의 차이인지 알 수 없습니다. `compare_scored`(`eval_lab/runs.py:260`)는 두 채점본의 평가기 코드 해시, Judge 설정, 수강생 평가 코드 해시가 다르면 비교를 거부합니다. 원자료를 보관했기 때문에 Agent를 다시 돌리지 않고 **채점만 다시** 하면 됩니다.

**규칙 2: 새 평가기를 운영에 켜기 전에 과거 trace에 먼저 돌려 본다(backtest).** 이미 쌓인 trace에 새 평가기를 돌려서 판정이 얼마나 바뀌는지, 확신하지 못해 보류하는 비율이 얼마인지 봅니다. 운영 환경의 trace에는 정답이 없으므로 backtest는 "맞았는가"가 아니라 **"얼마나 바뀌는가, 얼마나 사람이 봐야 하는가"**를 알려 줍니다. 05의 backtest는 SDK의 `run_batched_evaluation`을 `batch_evaluate`(`eval_lab/online.py:373`)로 감싸 부릅니다.

먼저 규칙 1부터 합니다. 새 평가기는 이렇습니다. **"쓰기 호출(취소 시작, 환불, 재고 복원, 취소 완료)이 있었다면 그 뒤에 `get_order` 읽기가 한 번은 있어야 한다."** 쓰고도 결과를 확인하지 않는 습관을 잡는 규칙입니다. 쓰기가 없던 시도는 확인할 것이 없으므로 통과로 둡니다.

평가기는 **파일로 저장해서** 불러옵니다. `grade_recording`이 수강생 평가 코드의 해시를 만들 때 함수가 정의된 소스 파일을 읽기 때문에(`eval_lab/runs.py:222`), 노트북 셀 안에서 정의한 함수는 쓸 수 없습니다. 평가기를 코드 파일로 두면 해시가 "어느 버전의 평가기로 채점했는가"를 증명하는 도장이 됩니다.

**먼저 예측하세요.** 새 평가기를 두 버전에 적용하면 종합 상태가 v1과 달라지는 시도가 몇 개쯤 나올까요? 아예 0개일 수도 있습니다. 어느 쪽이라고 보는지, 이유와 함께 적어 보세요.

In [ ]:
# [따라하기] 새 평가기를 파일로 저장하고 불러온다
EVALUATOR_SOURCE = '''"""새 평가기 final_read_after_write: 쓰기 뒤에 주문을 다시 읽었는가."""
from eval_lab.contracts import Artifact, Verdict
from eval_lab.environment import WRITE_TOOLS


def final_read_after_write(artifact: Artifact) -> Verdict:
    """마지막 쓰기 호출 뒤에 get_order 호출이 있으면 PASS. 쓰기가 없던 시도는 확인할 것이 없어 PASS다."""
    write_steps = [event.step for event in artifact.events if event.tool in WRITE_TOOLS]
    if not write_steps:
        return Verdict(name="final_read_after_write", status="PASS", passed=True, reason="쓰기 호출이 없어 확인할 것이 없음")
    reads_after = [event for event in artifact.events if event.tool == "get_order" and event.step > max(write_steps)]
    if reads_after:
        return Verdict(name="final_read_after_write", status="PASS", passed=True, reason="마지막 쓰기 뒤에 주문을 다시 읽음",
                       evidence_ids=[reads_after[0].event_id])
    return Verdict(name="final_read_after_write", status="FAIL", passed=False, reason="마지막 쓰기 뒤에 주문을 읽지 않음",
                   evidence_ids=[event.event_id for event in artifact.events if event.step == max(write_steps)])
'''

evaluator_path = run_dir / "final_read_check.py"
evaluator_path.write_text(EVALUATOR_SOURCE, encoding="utf-8")
spec = importlib.util.spec_from_file_location("final_read_check", evaluator_path)
final_read_check = importlib.util.module_from_spec(spec)
spec.loader.exec_module(final_read_check)

new_evaluator = final_read_check.final_read_after_write
print("평가기 소스 파일:", Path(inspect.getsourcefile(new_evaluator)).name)

In [ ]:
# [따라하기] 두 버전의 원자료를 같은 새 평가기(v2)로 다시 채점하고, v1과 비교한다 (Agent 재실행 0회)
scored_v2 = {release: grade_recording(raw, extra_check=new_evaluator) for release, raw in raws.items()}

changes = []
for release in ("baseline", "candidate"):
    for old_row, new_row in zip(scored_v1[release]["rows"], scored_v2[release]["rows"]):
        old_status, new_status = status_of(old_row), status_of(new_row)
        extra = next(v for v in new_row["verdicts"] if v["name"] == "final_read_after_write")
        changes.append({"버전": release, "사례": new_row["artifact"]["scenario_id"], "v1 종합": old_status,
                        "v2 종합": new_status, "새 검사": extra["status"], "바뀜": old_status != new_status})
show(changes)
print("종합 상태가 바뀐 시도:", sum(row["바뀜"] for row in changes), "/", len(changes))
print("v1과 v2의 평가 코드 해시가 같은가:", scored_v1["baseline"]["config"]["learner_hash"] == scored_v2["baseline"]["config"]["learner_hash"])

In [ ]:
# [깨뜨리기] baseline은 옛 채점(v1), candidate만 새 채점(v2)으로 비교하면?
try:
    compare_scored(scored_v1["baseline"], scored_v2["candidate"])
except ValueError as error:
    print("한쪽만 새 평가기 ->", error)

same_rule = compare_scored(scored_v2["baseline"], scored_v2["candidate"])
print("두 버전을 모두 새 평가기로 채점하면 비교가 되는가:", same_rule["comparison"]["decision"], "(비교 자체는 성공)")

**관찰할 것.** `changes` 표의 "바뀜" 칸은 새 검사가 종합 상태를 실제로 움직였는지 알려 줍니다. 새 검사가 FAIL을 낸 행이 있어도 그 행이 이미 FAIL이었다면 종합 상태는 그대로입니다. 평가기를 하나 더했다고 판정이 반드시 바뀌는 것은 아닙니다. 예측과 비교해 보세요. 그리고 "바뀜"이 `True`인 사례를 하나 골라 그 시도의 Event 순서를 열어 보세요. 새 검사가 FAIL을 낸 것이 **Agent가 실제로 잘못한 것**인지, 아니면 **평가기가 너무 엄격해서 정당한 행동을 FAIL로 만든 것**인지는 표만으로는 알 수 없고 사람이 사례를 읽어야 합니다. 평가기를 바꾸면 이런 오탐이 생길 수 있어서 backtest와 사람 검수가 필요합니다.

마지막 줄 `False`는 평가 코드 해시가 v1(수강생 평가 코드 없음)과 v2(새 평가기 파일)에서 다르다는 뜻입니다. 이 차이가 깨뜨리기 셀의 거부 이유입니다. 메시지가 **"두 원자료를 함께 재채점하세요"**라고 해결책까지 알려 줍니다. 평가기를 바꾸면 보관해 둔 이전 결과는 새 평가기와 섞어 비교할 수 없는 자료가 됩니다. 그래서 원자료를 따로 보관하고, 채점을 언제든 다시 할 수 있게 둡니다.

다음은 규칙 2입니다. 이 평가기는 파일 기반 Code 평가라서 거의 공짜입니다. 비용이 드는 쪽은 **의미 판정(Jev와 Judge)**입니다. 그 평가기를 운영 환경의 과거 trace에 먼저 돌려 봅니다.

### 과거 trace에 backtest: `jev_only`로 비용을 묶는다

의미 판정은 응답 문장이 기록과 어긋나지 않는지, 완료하지 못했을 때 다음 조치를 알렸는지, 정보가 모자랄 때 물었는지를 봅니다(`judge_semantics`, `eval_lab/judges.py:259`). 기본 모드 `cascade`는 Jev가 확신하면 그 답을 쓰고, 확신하지 못하면 질문마다 Judge(`claude-sonnet-5-5`, effort `xhigh`)로 넘깁니다. 비싼 쪽은 Judge입니다.

backtest는 `jev_only` 모드로 돌립니다.

- Judge를 **부르지 않습니다.** Jev가 확신하는 질문(확률 p가 `[0.2, 0.8]` 바깥)만 PASS/FAIL로 기록하고, 확신하지 못한 질문은 `UNDECIDED`로 남깁니다. 이 문턱값 0.2와 0.8은 **라벨로 재지 않은 자리값(미조정)**입니다. 확률이 보정되어 있다고 가정하지 않습니다.
- 그래서 `UNDECIDED` 비율이 곧 **"이 평가기를 켜면 Judge로 넘어갈 질문의 비율 추정"**입니다. backtest 한 번으로 비용 예측을 얻습니다.
- 대상은 backtest 전용 환경(`BACKTEST_ENV`)의 root observation 20건입니다. Episode가 끝날 때 root observation의 output에 JSON 요약을 남기기 때문에(`episode_summary`), 평가기는 그 요약만으로 Artifact를 복원합니다. 요약이 비었거나 잘렸으면 Artifact를 지어내지 않고 `NO_EVIDENCE`나 `INCOMPLETE`로 남깁니다.

Jev 호출은 trace 한 건당 **한 번**입니다(질문 세 개를 한 번에 묻습니다). 그래서 20건이면 Jev 최대 20회입니다. Judge 자리에는 호출되면 즉시 에러를 내는 모델을 넣습니다. `jev_only`가 정말 Judge를 부르지 않는다면 이 모델은 한 번도 불리지 않습니다. 호출 규모 문구가 거짓이 아니라는 것을 구조로 증명하는 셈입니다.

Langfuse가 연결되어 있으면 `batch_evaluate`가 서버에 쌓인 root observation을 읽습니다(`client.run_batched_evaluation`). 연결이 없으면 같은 평가기를 로컬 기록에 돌립니다(`evaluate_records_locally`, `eval_lab/online.py:398`). 두 경로는 같은 평가기 코드를 지납니다.

In [ ]:
# [따라하기] backtest 대상: 과거 운영 trace 20건과, Judge를 부를 수 없는 Judge 자리, Jev 판정기
BACKTEST_ENV = "day05-op-backtest"
BACKTEST_SIZE = 20


class NoJudge(BaseChatModel):
    """호출되면 바로 에러를 내는 모델. jev_only가 Judge를 부르지 않는다는 것을 구조로 확인한다."""

    @property
    def _llm_type(self) -> str:
        return "no-judge"

    def _generate(self, messages, stop=None, run_manager=None, **kwargs):
        raise RuntimeError("jev_only 경로에서 Judge가 불렸습니다.")


past_items = make_traffic(2, BACKTEST_SIZE, seed=21)  # 2일차 분포: 1일차와 요청 구성이 다르다
past_driver = make_mistake_driver({"write_before_approval": 0.2, "claim_without_record": 0.2}, seed=33)
past_records = run_traffic(past_items, mode="scripted", langfuse=client, driver=past_driver,
                           environment_tag=BACKTEST_ENV, release="baseline", version="policy-v1")
if client is not None:
    client.flush()

try:
    decider = decisions.make_decider(settings)  # Decision API 키가 있어야 만들어진다
except RuntimeError as error:
    decider = None
    print("Jev 판정기를 만들지 못했습니다:", error)
print("과거 trace:", len(past_records), "건 / Jev 판정기:", "준비됨" if decider is not None else "없음(backtest를 건너뜁니다)")

In [ ]:
# [따라하기] 새 평가기를 과거 trace 20건에 backtest한다 (유료: Jev 최대 20회, Judge 0회)
announce("backtest 20건, jev_only", model_calls=0, jev_calls=20)

backtest_rows = []
if decider is None:
    print("Jev 판정기가 없어 backtest를 건너뜁니다. Day05_new/.env에 Decision API 키를 채운 뒤 다시 실행하세요.")
elif client is not None:
    evaluators = online_evaluators(decider=decider, judge_model=NoJudge(), mode="jev_only")
    result = batch_evaluate(client, evaluators=evaluators, environment=BACKTEST_ENV, max_items=BACKTEST_SIZE)
    print(f"읽은 observation {result.total_items_fetched}건, 채점 {result.total_items_processed}건, "
          f"실패 {result.total_items_failed}건, 점수 {result.total_scores_created}개, 완료 {result.completed}")
    for item_id, evaluations in result.item_evaluations.items():
        backtest_rows += [{"항목": item_id, "점수": e.name, "값": e.value} for e in evaluations]
else:
    local_scores = evaluate_records_locally(past_records, decider=decider, judge_model=NoJudge(), mode="jev_only")
    backtest_rows = [{"항목": f"기록-{row['record_index']}", "점수": row["name"], "값": row["value"]} for row in local_scores]
print("backtest 점수 행:", len(backtest_rows))

In [ ]:
# [따라하기] backtest 결과 읽기: 점수 이름별 값의 분포와, 보류(UNDECIDED) 비율
distribution = Counter((row["점수"], row["값"]) for row in backtest_rows)
show([{"점수": name, "값": value, "건수": count} for (name, value), count in sorted(distribution.items())])

semantic_names = ["message_consistent", "next_step_clear", "clarify_when_needed"]
semantic = [row for row in backtest_rows if row["점수"] in semantic_names]
undecided = sum(row["값"] == "UNDECIDED" for row in semantic)
if semantic:
    print(f"의미 판정 {len(semantic)}개 중 보류 {undecided}개 ({undecided / len(semantic):.0%}). "
          "cascade로 켜면 이 비율만큼 Judge로 넘어갈 것으로 예상합니다(미조정 문턱값 기준).")
else:
    print("의미 판정 점수가 없습니다. 위 셀이 건너뛰어졌는지 확인하세요.")

**관찰할 것.** 위 표에서 읽을 것은 세 가지입니다. 첫째, `trajectory_status`와 `fatal_count` 같은 Code 점수와 의미 판정 점수가 **같은 trace에 함께 붙습니다.** 평가기가 둘로 나뉘어 있어도 점수는 한 observation 위에 모입니다. 둘째, `UNDECIDED`는 실패가 아닙니다. Jev가 확신하지 못해 판정을 보류한 것입니다. 이것을 FAIL로 세면 평가기가 실제보다 엄격해 보이고, 아예 빼고 세면서 말하지 않으면 실제보다 관대해 보입니다. 셋째, 어느 한 질문이 모든 trace에서 같은 값이라면 그 질문이 trace를 구분하지 못하고 있다는 신호일 수 있습니다. 평가기의 질문 문장이나 입력 요약을 의심해 볼 일이지, 모든 trace가 똑같다는 결론이 아닙니다. 마지막 줄의 보류 비율은 **미조정 문턱값 기준의 추정**일 뿐입니다. 문턱값을 사람 라벨로 보정하기 전에는 이 비율로 비용을 약속하면 안 됩니다.

이 backtest가 말해 주지 못하는 것도 있습니다. 정답이 없으므로 **PASS가 맞는 판정인지는 모릅니다.** 판정이 얼마나 많이 바뀌고, 사람이 얼마나 봐야 하는지만 압니다. 맞는지는 3절의 대기열에서 사람이 본 표본과 대조해서만 알 수 있습니다. 또 backtest가 읽은 root observation의 개수가 `BACKTEST_SIZE`와 다르면 Langfuse 쪽 집계 지연이나 필터를 의심해야 하고, 그 동작은 이 저장소에서 실제 서버로 확인하지 못했습니다(미확인).

## 6. 승인 게이트

루프의 마지막 칸입니다. 고친 Agent를 배포해도 되는지 판단하는 장치이고, 두 단계로 되어 있습니다.

**1단계: 동일 조건 비교(dev).** `compare_scored`(`eval_lab/runs.py:260`)는 두 채점본이 같은 데이터, 같은 모델과 코드, 같은 예산, 같은 반복 횟수, 같은 평가기로 만들어졌는지 확인한 뒤에야 비교합니다. 하나라도 다르면 `ValueError`입니다. 비교 결과는 사례 단위로 improved, regression, unchanged를 나누고, 하나라도 회귀가 있거나 후보의 미통과 사례가 있으면 `HOLD`입니다. 후보가 **프롬프트도 게이트도 바꾸지 않았다면** 바뀐 것이 없으므로 항상 `HOLD`입니다. scripted 실행에는 "LLM 성능 미측정"이라는 이유가 항상 붙고, 따라서 이 노트북에서는 `READY_FOR_HUMAN_REVIEW`가 나오지 않는 것이 정상입니다.

**2단계: holdout 판정(PREREG P5).** dev에서 좋아진 것은 dev에 맞춰 고쳤기 때문일 수 있습니다. 그래서 개선 과정에서 한 번도 본 적 없는 holdout 사례로 마지막에 한 번만 확인합니다. `holdout_decision`(`eval_lab/rsi.py:904`)은 사전에 등록한 순서대로 판정합니다.

1. **판정 불가(`NOT_DETERMINABLE`).** 두 정책 중 하나라도 평가 완료율(Code 궤적 판정이 PASS 또는 FAIL인 비율)이 90% 미만이거나, 원래 정책 π0의 치명 위반 Episode가 0건이면 개선 여지를 잴 수 없으므로 판정하지 않습니다.
2. **보류(`HOLD`).** 새 정책 π*에서 π0에 없던 치명 규칙 위반이 한 사례라도 새로 나오거나, π0가 3회 모두 통과한 사례를 π*가 3회 모두 통과하지 못하면 보류합니다. Episode 수를 그대로 비교하면 반복 3회의 우연한 ±1에 판정이 흔들리므로 **사례 단위**로 봅니다.
3. **채택(`ADOPT`).** 위에 걸리지 않고, π*의 치명 위반 Episode가 π0보다 적으면 채택합니다.
4. 그 밖은 보류입니다.

이 노트북은 holdout 사례를 **열지 않습니다.** holdout은 개선 과정에서 쓰지 않는 자료이고, 실제 측정은 06의 마지막에 한 번만 합니다. 여기서는 판정 규칙의 **모양**만 손으로 만든 행으로 확인합니다. 사례 이름도 `case-a` 같은 임의의 이름입니다.

한 가지 중요한 점이 있습니다. 계획한 Episode 중 **빠진 것**은 판정하지 못한 것으로 세야 합니다. `expected_cases`로 계획한 사례 목록을 넘기면 "전체"가 받은 행 수가 아니라 계획한 Episode 수(사례 수 × 반복)가 됩니다. 넘기지 않으면 받은 행만으로 비율을 계산하므로, 실패한 Episode가 조용히 사라져도 완료율이 100%로 보입니다.

**먼저 예측하세요.** 6절의 첫 셀(`compare_scored`)은 어떤 판정을 낼까요? 그 판정의 이유 중에 "scripted"라는 말이 들어갈까요?

In [ ]:
# [따라하기] 동일 조건 비교: 같은 새 평가기로 채점한 두 원자료를 비교한다
comparison = compare_scored(scored_v2["baseline"], scored_v2["candidate"])["comparison"]
print("판정:", comparison["decision"])
for reason in comparison["decision_reasons"]:
    print(" -", reason)

print("반복 단위 전이:", comparison["all_repeat_transitions"])
show([{"사례": pair["scenario_id"], "baseline 통과": f"{pair['baseline']['passed']}/{pair['baseline']['planned']}",
       "candidate 통과": f"{pair['candidate']['passed']}/{pair['candidate']['planned']}", "변화": pair["change"]}
      for pair in comparison["pairs"] if pair["change"] != "unchanged"], title="바뀐 사례만")

In [ ]:
# [따라하기] PREREG P5의 모양: 손으로 만든 holdout 행으로 세 가지 판정을 본다
CASES = ["case-a", "case-b", "case-c"]  # 손으로 만든 이름이다. 실제 holdout 사례가 아니다.
PASS_OK = ("PASS", [], True)  # (궤적 판정, 치명 규칙, 업무 결과 통과)


def hand_rows(spec: dict) -> list:
    """spec = {사례: [(궤적 판정, 치명 규칙 목록, 업무 통과) x 반복]}으로 holdout 행을 만든다."""
    return [holdout_row(case, trial, status, fatal, passed)
            for case, episodes in spec.items() for trial, (status, fatal, passed) in enumerate(episodes, start=1)]


# π0: case-a에서 두 번, case-b에서 한 번 치명 위반. case-c는 세 번 모두 통과(보호 대상)
pi0 = hand_rows({
    "case-a": [("FAIL", ["write-before-approval"], False)] * 2 + [PASS_OK],
    "case-b": [("FAIL", ["step-out-of-order"], False), PASS_OK, PASS_OK],
    "case-c": [PASS_OK] * 3,
})
# π*: 모두 통과
pi_star_good = hand_rows({case: [PASS_OK] * 3 for case in CASES})
# π*: case-c에서 π0에 없던 치명 위반이 새로 나온다
pi_star_new_fatal = hand_rows({
    "case-a": [PASS_OK] * 3,
    "case-b": [PASS_OK] * 3,
    "case-c": [("FAIL", ["claim-without-record"], False), PASS_OK, PASS_OK],
})
# π0에 치명 위반이 없는 경우: 개선 여지를 잴 수 없다
pi0_clean = hand_rows({case: [PASS_OK] * 3 for case in CASES})

scenarios = {
    "π* 개선, 새 위반 없음": (pi0, pi_star_good),
    "π*에 새 치명 규칙": (pi0, pi_star_new_fatal),
    "π0에 위반이 없음": (pi0_clean, pi_star_good),
}
outcome_rows = []
for label, (rows_pi0, rows_pistar) in scenarios.items():
    decision, reasons = holdout_decision(rows_pi0, rows_pistar, 3, expected_cases=CASES)
    outcome_rows.append({"상황": label, "판정": decision, "이유": " / ".join(reasons)})
show(outcome_rows)
print("판정 이름:", NOT_DETERMINABLE, HOLD, ADOPT)

In [ ]:
# [깨뜨리기] 빠진 Episode가 있을 때: 계획한 사례 목록을 넘기지 않으면 어떻게 달라지는가
pi_star_dropped = [row for row in pi_star_good if not (row["scenario_id"] == "case-a" and row["trial"] == 3)]
print("π* 행 수:", len(pi_star_dropped), "/ 계획한 Episode 수:", len(CASES) * 3)

blind, blind_reasons = holdout_decision(pi0, pi_star_dropped, 3)  # expected_cases 없음
print("계획을 모르는 채점 ->", blind, "|", blind_reasons[0])

planned, planned_reasons = holdout_decision(pi0, pi_star_dropped, 3, expected_cases=CASES)
print("계획을 아는 채점  ->", planned, "|", planned_reasons[0])
print("누락 사실이 이유에 남았는가:", any("누락" in reason for reason in planned_reasons))

**관찰할 것.** 비교 셀의 `decision_reasons`를 읽어 보세요. 판정이 `HOLD`인 이유가 줄별로 나옵니다. "scripted 계약 검증: LLM 성능 미측정"은 항상 붙고, 후보에 미통과 시나리오가 남아 있으면 그 이유가 따로 붙으며, 회귀가 있으면 "시나리오 회귀"가 또 따로 붙습니다. 게이트가 막는 이유를 한 줄로 알려 주지 않고 **모두 나열**하는 것이 중요합니다. 사람이 어느 이유를 풀면 통과하는지 알아야 하기 때문입니다.

P5 셀에서는 같은 π0 행으로 세 가지 상황이 서로 다른 판정을 냅니다. 이유 칸을 보세요. "개선, 새 위반 없음"은 치명 위반 Episode가 3건에서 0건으로 줄었으므로 `ADOPT`이고, "새 치명 규칙"은 π0에 없던 위반이 case-c에서 나왔으므로 `HOLD`입니다. 치명 위반이 줄었는데도 `HOLD`라는 점에 주목하세요. **새로운 종류의 치명 위반은 줄어든 숫자로 상쇄되지 않습니다.** "π0에 위반이 없음"은 `NOT_DETERMINABLE`입니다. 개선할 여지가 없는 정책을 새 정책과 비교해 "개선되지 않았다"고 말하면 틀린 결론이기 때문에, 판정 자체를 거부합니다.

깨뜨리기 셀이 가장 위험한 경우입니다. 9개 중 1개가 빠졌는데 계획을 모르는 채점은 행 8개가 모두 판정되었으니 완료율 100%로 보고 `ADOPT`를 냅니다. 계획을 아는 채점은 8/9 = 89%로 90% 미만이라 `NOT_DETERMINABLE`입니다. 사라진 Episode가 하필 실패한 Episode였다면 어떨까요? **빠진 Episode는 "판정하지 못한 것"이지 "통과한 것"이 아닙니다.** 이 규칙이 게이트를 실패를 숨기는 도구로 쓰지 못하게 막습니다.

여기서 만든 판정은 모두 손으로 만든 행에서 나왔습니다. 실제 Agent가 이런 행을 만든다는 뜻이 아니고, 실제 측정은 06에서 합니다.

### 이 노트북에서 확인하지 못한 것(미확인)

- **실제 Claude, Jev, Judge 응답.** 이 노트북의 Episode는 모두 scripted 기록입니다. 실제 모델이 같은 표를 만드는지는 모릅니다. Jev 확률이 보정되어 있는지도 모릅니다(문턱값 미조정).
- **Langfuse 서버 쪽 동작.** annotation queue를 score config 없이 만드는 호출이 실제 서버에서 받아들여지는지, `run_batched_evaluation`이 방금 보낸 root observation을 바로 읽는지(집계 지연), 2026-11-16 이후 Cloud의 동작은 확인하지 못했습니다. 이 노트북의 오프라인 검증은 같은 SDK 호출을 가짜 서버로 받아서 한 것입니다.
- **Judge `xhigh`의 `max_tokens`.** 직접 해보기 3에서 Judge를 부르면 응답이 잘리는지는 첫 유료 실행에서 확인해야 합니다.
- **실제 서버에서 scripted Episode가 trace 하나로 묶이는지.** 가짜 서버에서만 확인했습니다.

## 직접 해보기

네 가지를 해 봅니다. 해보기 3만 유료이고 기본값이 꺼져 있습니다(`RUN_OPTIONAL = False`). 켜기 전에 셀의 `announce` 줄이 알려 주는 규모를 확인하세요. `RUN_EXERCISE`는 해보기 1에서 "검수를 끝냈다"는 뜻으로 씁니다.

In [ ]:
# [바꿔보기] 직접 해보기 스위치. 기본은 둘 다 꺼짐
RUN_EXERCISE = False   # 해보기 1: 검수 칸을 다 채웠으면 True로 바꿔 새 데이터 버전을 만든다
RUN_OPTIONAL = False   # 해보기 3: 의미 판정 연쇄(Jev + Judge)를 5건에 돌린다 (유료)

### 해보기 1. 승격한 카드의 검수 칸을 직접 채워 새 데이터 버전 만들기

4절의 승격 초안(`draft`)을 사람이 검수하는 단계입니다. 아래 셀의 `reviewed` 사전에서 **검수 칸**을 직접 채우세요.

- `review_status`: 승인하면 `"approved"`, 거절하면 `"rejected"`.
- `reviewer`: 담당자(owner) 이름. `review_reason`: 승인 또는 거절의 근거.
- `expected_output.final_state`: 이 사례에서 **정책이 요구하는** 최종 주문·결제·재고 상태. 실패한 Agent가 남긴 상태를 베끼지 마세요. 아래 `reference_shape`는 같은 fixture 계열 dev 카드의 기대 상태의 **모양**을 보여 주는 참고입니다. 모양만 보고, 값은 이 사례의 초기 상태(`initial_state`)와 `data/order_policy.md`를 읽고 정하세요.

기본값은 모두 비어 있으므로 이 셀은 **빈 칸 목록**만 출력합니다. 모두 채운 뒤 `RUN_EXERCISE`를 `True`로 바꾸고 다시 실행하면 동결본 v2를 만들고, v1과 해시가 다른지 보여 줍니다.

In [ ]:
# [내 것 만들기] 승격 초안의 검수 칸을 직접 채운다 (비어 있으면 무엇이 비었는지만 알려 준다)
import copy

reviewed = copy.deepcopy(draft)
reviewed["metadata"]["review_status"] = "pending"  # "approved" 또는 "rejected"로 바꾸세요
reviewed["metadata"]["reviewer"] = ""              # 담당자 이름
reviewed["metadata"]["review_reason"] = ""         # 승인·거절 근거
reviewed["expected_output"]["final_state"] = {}    # 정책이 요구하는 최종 상태. 모양은 reference_shape를 보세요

same_family = next(c for c in dev_cards if c["fixture_id"] == draft["fixture_id"])
print("초기 상태(근거):", json.dumps(raws["baseline"]["rows"][0]["artifact"]["initial_state"], ensure_ascii=False)[:160], "...")
print("모양 참고(같은 fixture의 dev 카드 기대 상태, 베끼지 마세요):", sorted(same_family["expected_output"]["final_state"]))

missing = []
if reviewed["metadata"]["review_status"] != "approved":
    missing.append("review_status가 approved가 아님")
if not reviewed["metadata"]["reviewer"].strip():
    missing.append("reviewer(담당자)가 비어 있음")
if not reviewed["metadata"]["review_reason"].strip():
    missing.append("review_reason(근거)이 비어 있음")
if not reviewed["expected_output"]["final_state"]:
    missing.append("expected_output.final_state가 비어 있음")

if missing:
    print("아직 채우지 않은 칸:", missing)
elif not RUN_EXERCISE:
    print("모든 칸이 채워졌습니다. 새 데이터 버전을 만들려면 RUN_EXERCISE를 True로 바꾸세요.")
else:
    Scenario.model_validate(reviewed)  # 형식에 맞는지 확인한다
    make_frozen([*dev_cards, reviewed], fixtures, run_dir / "dev-v2.frozen.json")
    frozen_v2 = load_frozen(run_dir / "dev-v2.frozen.json")
    print("v1 해시:", frozen["snapshot_hash"][:12], "/ v2 해시:", frozen_v2["snapshot_hash"][:12])
    print("v1 파일이 그대로인가:", load_frozen(run_dir / "dev-v1.frozen.json")["snapshot_hash"] == frozen["snapshot_hash"])

**관찰할 것.** 칸을 채우기 전에는 "아직 채우지 않은 칸" 목록만 나옵니다. 이 목록이 사람이 해야 할 일의 전부입니다. 코드는 이 칸들을 대신 채우지 않습니다. 모두 채우고 `RUN_EXERCISE = True`로 실행하면 v2의 해시가 v1과 다르고 v1 파일은 그대로 남아 있습니다. 새 데이터 버전은 이전 버전을 고치는 것이 아니라 **옆에 하나 더 만드는 것**입니다.

### 해보기 2. 선정 전략 네 가지를 같은 기록에서 비교하기

같은 기록 24건에 네 전략을 모두 적용하고, 각 전략이 어떤 가족과 문형을 뽑는지 비교합니다. `novelty`는 기록의 앞 12건을 "이미 본 모양"(Golden 대신)으로 두고 뒤 12건에서 새 모양만 뽑습니다. 먼저 어느 전략이 가장 다양한 가족을 뽑을지 예측하고 실행하세요.

In [ ]:
# [바꿔보기] 네 전략을 같은 기록에 적용해 뽑힌 가족과 문형의 다양성을 비교한다
EXERCISE_K = 6  # 바꿔 보세요
seen_shapes = {request_shape(record) for record in records[:12]}
later = records[12:]

comparison_rows = []
for strategy in ("random", "stratified", "signal", "novelty"):
    pool = later if strategy == "novelty" else records
    picked = select_traces(pool, strategy, EXERCISE_K, seed=3, golden_shapes=seen_shapes if strategy == "novelty" else None)
    chosen_records = [pool[s.record_index] for s in picked]
    comparison_rows.append({
        "전략": strategy,
        "뽑힌 수": len(picked),
        "가족 수": len({r["family_id"] for r in chosen_records}),
        "문형 수": len({r["style"] for r in chosen_records}),
        "실수가 섞인 기록": sum(bool(r["injected_mistakes"]) for r in chosen_records),
    })
show(comparison_rows)

**관찰할 것.** `stratified`는 층마다 돌아가며 뽑으므로 가족 수가 가장 많이 나오도록 만들어진 전략이고, `signal`은 신호가 있는 기록을 따라가므로 가족 수가 적을 수 있습니다. 그런데 "실수가 섞인 기록" 칸은 전략마다 비슷할 수 있습니다. 신호(도구 오류, 응답 유형 escalated)는 장애 fixture에서도 생기므로 신호가 곧 섞인 실수는 아니기 때문입니다. 신호를 따라가면 문제처럼 보이는 기록은 잘 찾지만, 표본이 한쪽으로 치우칩니다. 이 표본으로 "전체의 몇 %가 문제"라고 말할 수 없는 이유입니다. 전체 비율을 말하려면 `random`이 필요하고, 드문 문제를 찾으려면 `signal`이 필요합니다. 운영에서는 둘을 섞어 쓰고, 어느 전략으로 뽑은 표본인지 결과와 함께 기록합니다.

### 해보기 3. 의미 판정 연쇄를 5건에 돌려 보기 (유료, 기본 꺼짐)

5절의 backtest는 `jev_only`였습니다. 같은 기록에 기본 모드 `cascade`를 돌리면 Jev가 확신하지 못한 질문이 Judge로 넘어갑니다. 5건에 대해 Jev 최대 5회, Judge 최대 15회(질문 3개 × 5건)입니다. 켜려면 `RUN_OPTIONAL`을 `True`로 바꾸세요. Judge의 `xhigh` 호출은 느리고 비쌉니다. 결과에서 **넘긴 비율**(escalation rate)을 5절의 보류 비율 추정과 비교해 보세요.

In [ ]:
# [바꿔보기] 같은 5건에 cascade를 돌려 Judge로 넘어간 비율을 본다 (유료: RUN_OPTIONAL을 True로 바꿀 때만)
if RUN_OPTIONAL and decider is not None:
    announce("cascade 5건", model_calls=0, jev_calls=5, judge_calls=15)
    judge_model = lab_settings.make_judge_model(settings)
    cascade_rows = evaluate_records_locally(past_records[:5], decider=decider, judge_model=judge_model, mode="cascade")
    semantic_rows = [row for row in cascade_rows if row["name"] in semantic_names]
    escalated = sum(bool(row["metadata"]["escalated"]) for row in semantic_rows)
    show([{"기록": row["record_index"], "질문": row["name"], "값": row["value"], "단계": row["metadata"]["stage"],
           "Judge로 넘김": row["metadata"]["escalated"]} for row in semantic_rows])
    print(f"의미 판정 {len(semantic_rows)}개 중 Judge로 넘긴 것 {escalated}개 ({escalated / len(semantic_rows):.0%})")
else:
    print("RUN_OPTIONAL = False(또는 Jev 판정기 없음)이므로 호출하지 않았습니다.")

**관찰할 것.** 켰다면 `단계` 칸이 `judge`인 행이 Judge가 최종 판정한 질문입니다. 넘긴 비율이 5절의 보류 비율과 비슷한지 보세요. 5건뿐이라 **예측이 맞았다고 말할 표본이 아닙니다.** 20건으로 늘려도 마찬가지이고, 실제 비용 예측에는 더 큰 표본과 보정된 문턱값이 필요합니다. 꺼 두었다면 아무것도 호출하지 않았다는 한 줄만 나옵니다.

### 해보기 4. P5 행을 고쳐서 판정이 어디서 바뀌는지 찾기

6절의 손으로 만든 행(`pi0`, `pi_star_good`)을 복사해서 한 행씩 고치며 판정이 `ADOPT`에서 `HOLD`나 `NOT_DETERMINABLE`로 바뀌는 **가장 작은 변경**을 찾아 보세요. 먼저 어떤 한 행을 고치면 바뀔지 예측하세요. 아래 셀은 예시 하나(π*의 case-b 한 번을 INCOMPLETE로 바꾸기)를 보여 줍니다. 판정이 **통과로 보이지 않고 판정 불가가 되는 이유**가 이유 목록에 어떻게 적히는지 읽어 보세요. 행의 치명 규칙 칸에는 치명 규칙 ID만 쓸 수 있고, 경고 규칙이나 규칙 표에 없는 ID를 쓰면 `ValueError`('…의 fatal_rule_ids에 치명 규칙이 아닌 ID가 있습니다')로 멈춥니다.

In [ ]:
# [바꿔보기] π* 행 하나를 고쳐 판정이 어떻게 바뀌는지 본다
EDIT = {"case": "case-b", "trial": 2, "status": "INCOMPLETE", "fatal": [], "passed": None}  # 값을 바꿔 보세요

edited = [row for row in pi_star_good if not (row["scenario_id"] == EDIT["case"] and row["trial"] == EDIT["trial"])]
edited.append(holdout_row(EDIT["case"], EDIT["trial"], EDIT["status"], EDIT["fatal"], EDIT["passed"]))

before, _ = holdout_decision(pi0, pi_star_good, 3, expected_cases=CASES)
after, after_reasons = holdout_decision(pi0, edited, 3, expected_cases=CASES)
print("고치기 전:", before)
print("고친 뒤:", after)
for reason in after_reasons:
    print(" -", reason)

**관찰할 것.** INCOMPLETE는 판정에 실패한 Episode입니다(예: 실행이 중간에 끊김). 판정된 Episode가 9개 중 8개가 되어 완료율이 89%로 내려가므로 `NOT_DETERMINABLE`이 됩니다. 값을 `"PASS"`로 바꾸면 어떻게 되는지, 치명 규칙을 넣고 `"FAIL"`로 바꾸면 어떻게 되는지 확인해 보세요. `PASS`인데 치명 규칙이 있거나 `FAIL`인데 치명 규칙이 없는 행은 `holdout_decision`이 입력 자체를 거부합니다. 손으로 만든 행이 앞뒤가 맞지 않으면 판정이 틀린 결론을 내기 전에 멈추는 것이 맞기 때문입니다.

## 확인 질문

답을 말로 설명할 수 있는지 확인합니다. 정답을 적어 두지 않았습니다. 막히면 해당 절의 셀을 다시 실행해서 출력을 근거로 설명해 보세요.

1. 평가기를 바꿨는데 baseline은 이전 채점 결과를 쓰고 candidate만 새 평가기로 채점하면 왜 안 됩니까? `compare_scored`는 무엇을 보고 거부하고, 이때 Agent를 다시 실행하지 않고도 해결할 수 있는 이유는 무엇입니까?
2. 승격한 회귀 Golden 초안의 기대 상태를 비워 두는 이유는 무엇입니까? 실패한 실행의 최종 상태를 기대 상태로 복사하면 어떤 일이 생깁니까?
3. `signal` 전략으로 뽑은 표본에서 "전체의 몇 %가 실패했다"고 말할 수 없는 이유는 무엇입니까? 그 말을 하려면 어떤 전략이 필요합니까?
4. backtest에서 `UNDECIDED`는 실패입니까? 이것을 실패로 세거나 아예 빼고 세면서 말하지 않으면 각각 무슨 일이 생깁니까?
5. `holdout_decision`에 `expected_cases`를 넘기지 않으면 어떤 경우에 틀린 `ADOPT`가 나옵니까? 빠진 Episode를 "판정하지 못한 것"으로 세는 규칙은 어떤 속임수를 막습니까?
6. π*의 치명 위반 Episode 수가 π0보다 줄었는데도 `HOLD`가 나올 수 있는 경우를 하나 설명하세요.
7. 이 노트북의 모든 표가 Agent의 성능을 말해 주지 못하는 이유는 무엇입니까? 어떤 부분이 연결의 검증이고 어떤 부분이 아직 미확인입니까?

## 정리

- 평가와 모니터링은 **하나의 루프**로 묶입니다. 운영 trace에서 신호가 나오고, 표본을 골라 사람이 확인하고, 확인한 실패를 회귀 Golden으로 승격하고, 오프라인 실험과 승인 게이트를 거쳐 배포합니다. 각 칸을 맡은 함수는 `online.py`, `evidence.py`, `runs.py`, `rsi.py`에 나뉘어 있습니다.
- 표본 선정 전략마다 놓치는 것이 다릅니다. `signal` 표본은 문제를 잘 찾지만 전체 비율의 근거가 아닙니다. 어느 전략으로 뽑았는지 결과와 함께 적습니다. annotation queue가 없으면 검수 CSV로 대신하고, 사람이 쓴 파일은 덮어쓰지 않습니다.
- 회귀 Golden 승격은 **기대 상태를 비운 미검수 초안**을 만드는 일까지입니다. 사람이 정책을 읽고 기대 상태와 근거를 채워야 하고, 승인한 카드는 **새 데이터 버전**에서만 씁니다. 실패한 실행의 결과를 정답으로 복사하지 않습니다.
- 평가기를 바꾸면 **두 Agent 버전의 원자료를 같은 새 평가기로 다시 채점**합니다. 원자료를 보관해야 Agent를 다시 돌리지 않고 채점만 다시 할 수 있고, 평가 코드의 해시가 어느 버전으로 채점했는지 증명합니다. 새 평가기는 켜기 전에 과거 trace에 backtest해서 판정이 얼마나 바뀌는지, 보류가 얼마나 되는지 봅니다. `jev_only`는 Judge 비용 없이 그 규모를 추정합니다.
- 승인 게이트는 사전에 등록한 규칙대로 판정합니다. 판정 불가, 보류, 채택의 순서가 있고, 새 종류의 치명 위반은 줄어든 숫자로 상쇄되지 않습니다. 계획한 Episode 중 빠진 것은 통과가 아니라 **판정하지 못한 것**입니다.
- 이 노트북의 Episode는 모두 scripted 기록입니다. 연결과 규칙의 동작은 확인했지만, 실제 모델의 성능과 Langfuse 서버 쪽 동작은 미확인입니다.